# Capítulo 4 — Métodos Não Paramétricos

**Livro:** começa na página **71** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
set.seed(42)
options(digits = 4, repr.plot.width = 7, repr.plot.height = 4.5)
suppressPackageStartupMessages({ library(glmnet); library(rpart); library(randomForest); library(FNN); library(e1071) })
cat("Pronto.", R.version.string, "
")

## Splines · PDF p. 74

#### Exercício: Splines de suavização

Ajuste `smooth.spline` aos dados do seno com `df = 4`, `df = 10` e com o `df` escolhido por validação cruzada (padrão). Desenhe as três curvas.

In [ ]:
sim_seno <- function(n, sd = 0.3) {
  x <- runif(n)
  data.frame(x = x, y = sin(2 * pi * x) + rnorm(n, sd = sd))
}
d <- sim_seno(150)
plot(d, col = "grey60", pch = 16)
for (f in list(smooth.spline(d$x, d$y, df = 4), smooth.spline(d$x, d$y, df = 10), smooth.spline(d$x, d$y)))
  lines(predict(f, seq(0, 1, 0.01)), lwd = 2, col = f$df %/% 3 + 1)
smooth.spline(d$x, d$y)$df

## k Vizinhos Mais Próximos · PDF p. 75

#### Exercício: Escolhendo k no KNN

Use `FNN::knn.reg` nos dados do seno (200 de treino, 2.000 de teste) com k de 1 a 60. Desenhe o EQM de teste contra k e diga qual k é o melhor.

In [ ]:
sim_seno <- function(n, sd = 0.3) {
  x <- runif(n)
  data.frame(x = x, y = sin(2 * pi * x) + rnorm(n, sd = sd))
}
tr <- sim_seno(200); te <- sim_seno(2000)
ks <- 1:60
eqm <- sapply(ks, function(k) mean((te$y - knn.reg(tr["x"], te["x"], tr$y, k = k)$pred)^2))
plot(ks, eqm, type = "l", xlab = "k", ylab = "EQM de teste"); ks[which.min(eqm)]

## Nadaraya-Watson · PDF p. 77

#### Exercício: Nadaraya-Watson à mão

Implemente `nw(x0, x, y, h)` com núcleo gaussiano: média de y ponderada por `dnorm((x - x0) / h)`. Desenhe o estimador com h = 0,02 e h = 0,2. Qual fica muito ondulado e qual fica liso demais?

In [ ]:
sim_seno <- function(n, sd = 0.3) {
  x <- runif(n)
  data.frame(x = x, y = sin(2 * pi * x) + rnorm(n, sd = sd))
}
d <- sim_seno(150)
nw <- function(x0, x, y, h) sapply(x0, function(a) { w <- dnorm((x - a) / h); sum(w * y) / sum(w) })
grade <- seq(0, 1, 0.005)
plot(d, col = "grey60", pch = 16)
lines(grade, nw(grade, d$x, d$y, 0.02), col = "blue", lwd = 2)
lines(grade, nw(grade, d$x, d$y, 0.2), col = "red", lwd = 2)

## Regressão Polinomial Local · PDF p. 79

#### Exercício: loess

Compare `loess(y ~ x, span = 0.2)` com `span = 0.75` nos dados do seno. O que o `span` controla?

In [ ]:
sim_seno <- function(n, sd = 0.3) {
  x <- runif(n)
  data.frame(x = x, y = sin(2 * pi * x) + rnorm(n, sd = sd))
}
d <- sim_seno(150); grade <- data.frame(x = seq(0, 1, 0.01))
plot(d, col = "grey60", pch = 16)
lines(grade$x, predict(loess(y ~ x, d, span = 0.2), grade), col = "blue", lwd = 2)
lines(grade$x, predict(loess(y ~ x, d, span = 0.75), grade), col = "red", lwd = 2)
# span = fração dos pontos usada em cada ajuste local: maior span, curva mais lisa

### Exemplo 1: Kernel Ridge Regression · PDF p. 85

#### Exercício: Kernel ridge regression do zero

Com núcleo gaussiano K(a, b) = exp(−(a−b)²/(2·0,1²)), calcule α = (K + λI)⁻¹y com λ = 0,1 e preveja ŷ(x₀) = Σ αᵢ K(x₀, xᵢ) numa grade. Desenhe o resultado.

In [ ]:
sim_seno <- function(n, sd = 0.3) {
  x <- runif(n)
  data.frame(x = x, y = sin(2 * pi * x) + rnorm(n, sd = sd))
}
d <- sim_seno(100)
nucleo <- function(a, b, s = 0.1) exp(-outer(a, b, "-")^2 / (2 * s^2))
alfa <- solve(nucleo(d$x, d$x) + 0.1 * diag(nrow(d)), d$y)
grade <- seq(0, 1, 0.01)
plot(d, col = "grey60", pch = 16); lines(grade, nucleo(grade, d$x) %*% alfa, col = "blue", lwd = 2)

### Exemplo 2: Support Vector Regression Machine · PDF p. 91

#### Exercício: Support vector regression

Ajuste `e1071::svm(y ~ x, type = "eps-regression")` aos dados do seno e calcule o EQM de teste. Quantos vetores de suporte o modelo usou?

In [ ]:
sim_seno <- function(n, sd = 0.3) {
  x <- runif(n)
  data.frame(x = x, y = sin(2 * pi * x) + rnorm(n, sd = sd))
}
tr <- sim_seno(200); te <- sim_seno(2000)
m <- svm(y ~ x, data = tr, type = "eps-regression")
c(EQM = mean((te$y - predict(m, te))^2), vetores_suporte = m$tot.nSV)

## Modelos Aditivos · PDF p. 92

#### Exercício: Modelo aditivo (GAM)

Com `mgcv::gam`, ajuste `Ozone ~ s(Temp) + s(Wind)` em `airquality` e desenhe os efeitos de cada variável. Algum efeito é claramente não linear?

In [ ]:
library(mgcv)
m <- gam(Ozone ~ s(Temp) + s(Wind), data = airquality)
summary(m)$s.table
par(mfrow = c(1, 2)); plot(m, shade = TRUE); par(mfrow = c(1, 1))

## Árvores de Regressão · PDF p. 95

#### Exercício: Árvore e poda

Ajuste `rpart(Ozone ~ ., airquality)`, desenhe a árvore (`plot` + `text`) e pode-a com o `cp` de menor erro de validação cruzada (`printcp`).

In [ ]:
arv <- rpart(Ozone ~ ., data = airquality)
printcp(arv)
cp_bom <- arv$cptable[which.min(arv$cptable[, "xerror"]), "CP"]
podada <- prune(arv, cp = cp_bom)
plot(podada, margin = 0.1); text(podada, cex = 0.8)

## Bagging e Florestas Aleatórias · PDF p. 100

#### Exercício: Bagging feito à mão

Separe `airquality` (sem NA) em treino e teste. Ajuste 100 árvores em amostras *bootstrap* do treino e faça a média das previsões. Compare o EQM de teste com o de uma árvore só.

In [ ]:
aq <- na.omit(airquality); i <- sample(nrow(aq), 80)
tr <- aq[i, ]; te <- aq[-i, ]
uma <- mean((te$Ozone - predict(rpart(Ozone ~ ., tr), te))^2)
prev <- replicate(100, predict(rpart(Ozone ~ ., tr[sample(nrow(tr), replace = TRUE), ]), te))
c(uma_arvore = uma, bagging = mean((te$Ozone - rowMeans(prev))^2))

### Florestas Aleatórias · PDF p. 103

#### Exercício: Floresta aleatória e importância

Ajuste `randomForest(Ozone ~ ., na.omit(airquality), importance = TRUE)`. Mostre o erro *out-of-bag* e o gráfico de importância (`varImpPlot`).

In [ ]:
rf <- randomForest(Ozone ~ ., data = na.omit(airquality), importance = TRUE)
rf
varImpPlot(rf)

## Boosting · PDF p. 107

#### Exercício: Boosting com gbm

Com `gbm` (distribuição gaussiana, 2.000 árvores, `shrinkage = 0.01`, `interaction.depth = 2`) nos dados do seno, desenhe o EQM de teste contra o número de árvores.

In [ ]:
sim_seno <- function(n, sd = 0.3) {
  x <- runif(n)
  data.frame(x = x, y = sin(2 * pi * x) + rnorm(n, sd = sd))
}
library(gbm)
tr <- sim_seno(300); te <- sim_seno(2000)
m <- gbm(y ~ x, data = tr, distribution = "gaussian", n.trees = 2000, shrinkage = 0.01, interaction.depth = 2)
arvores <- seq(50, 2000, 50)
eqm <- sapply(arvores, function(k) mean((te$y - predict(m, te, n.trees = k))^2))
plot(arvores, eqm, type = "l"); arvores[which.min(eqm)]

## Redes Neurais Artificiais · PDF p. 109

#### Exercício: Rede neural com uma camada

Com `nnet` (10 neurônios, `linout = TRUE`, `decay = 1e-3`, `maxit = 2000`), ajuste os dados do seno e desenhe a curva aprendida.

In [ ]:
sim_seno <- function(n, sd = 0.3) {
  x <- runif(n)
  data.frame(x = x, y = sin(2 * pi * x) + rnorm(n, sd = sd))
}
library(nnet)
d <- sim_seno(200)
rede <- nnet(y ~ x, data = d, size = 10, linout = TRUE, decay = 1e-3, maxit = 2000, trace = FALSE)
grade <- data.frame(x = seq(0, 1, 0.01))
plot(d, col = "grey60", pch = 16); lines(grade$x, predict(rede, grade), col = "blue", lwd = 2)